### Introduction to Vehicle Routing Optimization (VRP) with NVIDIA cuOpt

NVIDIA cuOpt is widely recognized as the world's fastest logistics and decision optimization engine. While traditional CPU-based solvers can take minutes or even hours to calculate complex routing matrices as the number of stops increases, cuOpt leverages the massive parallel computing power of GPUs to discover globally optimized routes in a fraction of a second.
In this section of the notebook, we transition from conventional linear resource allocation models (MIP) to the Capacitated Vehicle Routing Problem (CVRP), applied directly to agricultural supply chain and crop logistics.

#### Supply Delivery with a Heterogeneous Fleet 

This code aims to plan deliveries using two trucks with different capacities, minimizing the total travel cost while respecting each truck’s load limit.

This is called a Heterogeneous Vehicle Routing Problem (VRP) because the vehicles have different characteristics—in this case, capacities of 250 and 100 bags.

#### Libraries

In [22]:
from cuopt import routing
import numpy as np
import cudf

#### Creating Synthetic Data and GPU Spatial Matrix Calculation

In [44]:
# Initialize random seed for reproducibility
np.random.seed(42) 
n_fazendas = 15

In [45]:
#Generate random geographical coordinates representing the agricultural zone
lats = np.random.uniform(low=-23.10, high=-22.70, size=n_fazendas)
lons = np.random.uniform(low=-47.26, high=-46.86, size=n_fazendas)

# Override index 0 to act as our Central Silo (Depot)
lats[0], lons[0] = -22.90, -47.06 # Silo Central 

#Create the GPU DataFrame using RAPIDS cuDF
df_coords = cudf.DataFrame({'id_local': range(n_fazendas), 'latitude': lats, 'longitude': lons,
'bag_demand': np.random.randint(10, 26, size=n_fazendas) })

#The central depot requires zero pickup demand
df_coords.loc[0, 'demanda_sacos'] = 0 

In [46]:
#High-performance GPU function to calculate the cost matrix
def calcular_matriz_km(df):
    coords = cp.array(df[['latitude', 'longitude']].to_pandas().values)
    coords_km = coords * 111.0 
    dist_matrix = cp.sqrt(cp.sum((coords_km[:, None, :]- coords_km[None, :, :]) ** 2, axis=-1))
    return cudf.DataFrame(dist_matrix.get().astype(np.float32))

In [47]:
#Run the function to build our cost baseline
matrix_custo = calcular_matriz_km(df_coords)

In [48]:
print("--- GPU GEOPROCESSING SUCCESSFUL ---")
print(f"Distance Matrix Shape: {matrix_custo.shape} (15x15 nodes)")
print("Data structure is fully allocated on the GPU and optimized for cuOpt.")

--- GPU GEOPROCESSING SUCCESSFUL ---
Distance Matrix Shape: (15, 15) (15x15 nodes)
Data structure is fully allocated on the GPU and optimized for cuOpt.


In [49]:
from cuopt import routing
import cudf
import numpy as np

# 1. Locais, pedidos e veículos
num_locations = len(df_coords)
num_vehicles = 2
num_orders = num_locations - 1  # Exclui o depósito

# 2. Demandas dos clientes e capacidades dos caminhões
# A demanda deve seguir a mesma ordem dos pedidos.
farm_demands = cudf.Series(
    df_coords['bag_demand'].iloc[1:].values,
    dtype=np.int32,
)

fleet_capacities = cudf.Series([250, 100], dtype=np.int32)

# Verificação necessária para uma carga por caminhão
total_demand = int(farm_demands.sum())
total_capacity = int(fleet_capacities.sum())

print(f"Demanda total: {total_demand} sacos")
print(f"Capacidade total: {total_capacity} sacos")

if total_demand > total_capacity:
    raise ValueError(
        "A demanda excede os 350 sacos disponíveis. "
        "Aumente a capacidade ou a frota, ou revise os pedidos."
    )

# 3. Modelo de roteamento
data_model = routing.DataModel(
    num_locations,
    num_vehicles,
    num_orders,
)

data_model.add_cost_matrix(matrix_custo)

# 4. Ambos os caminhões começam e terminam no depósito 0
data_model.set_vehicle_locations(
    cudf.Series([0, 0], dtype=np.int32),
    cudf.Series([0, 0], dtype=np.int32),
)

# 5. Pedidos nos locais 1 até o último local
data_model.set_order_locations(
    cudf.Series(range(1, num_locations), dtype=np.int32)
)

# 6. Limites de carga por caminhão
data_model.add_capacity_dimension(
    "bags",
    farm_demands,
    fleet_capacities,
)

# 7. Execução
settings = routing.SolverSettings()
settings.set_time_limit(5.0)

solution = routing.Solve(data_model, settings)

# 8. Resultados
status = solution.get_status()
print(f"Status: {status}")

if status == 0:
    print(
        f"Distância total: "
        f"{solution.get_total_objective():.2f} km"
    )

    final_routes = solution.get_route()
    print(final_routes)
else:
    print("O solver não retornou uma solução com sucesso.")

Demanda total: 231 sacos
Capacidade total: 350 sacos
Status: 0
Distância total: 138.70 km
    route  arrival_stamp  truck_id  location      type
0       0            0.0         0         0     Depot
1      12            0.0         0        13  Delivery
2       4            0.0         0         5  Delivery
3       9            0.0         0        10  Delivery
4       3            0.0         0         4  Delivery
5       5            0.0         0         6  Delivery
6      13            0.0         0        14  Delivery
7       7            0.0         0         8  Delivery
8       6            0.0         0         7  Delivery
9      10            0.0         0        11  Delivery
10      0            0.0         0         1  Delivery
11     11            0.0         0        12  Delivery
12      1            0.0         0         2  Delivery
13      8            0.0         0         9  Delivery
14      2            0.0         0         3  Delivery
15      0            0.0      

O solver encontrou uma solução com sucesso (Status: 0) e usou apenas o caminhão grande.
Como a demanda total é de 231 sacos, ela cabe inteiramente no caminhão de 250 sacos. Disponibilizar dois veículos não obriga o solver a usar ambos.

Ela atende às 14 fazendas, percorre 138,70 km segundo a matriz calculada e deixa 19 sacos de capacidade disponível.
As colunas significam:
Coluna	Significado
route	Identificador do pedido; não indica a ordem das visitas.

arrival_stamp	Informação de chegada; os zeros não representam tempos reais de viagem, pois o modelo não recebeu uma matriz de tempos.

truck_id	Veículo utilizado: 0 é o caminhão grande.

location	Índice do local visitado. Leia essa coluna de cima para baixo para acompanhar a rota.

type	Depósito (Depot) ou entrega (Delivery).